In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("Alles erfolgreich geladen!")

Alles erfolgreich geladen!


In [15]:
import numpy as np
import pandas as pd

# ==========================================
# 1. GESETZLICHE PARAMETER (STAND 2026) & INPUTS
# ==========================================

# Stipendium (SDW) / BAföG-Sätze
BAFOEG_ERSATZ_MAX = 855.0  # Maximaler BAföG-Ersatzbetrag
BUCHGELD = 300.0  # Einkommensunabhängiges Buchgeld

# BAföG-Anrechnungslogik (§ 21, 23 BAföG)
WERBUNGSKOSTEN_PAUSCHALE_MONAT = 102.50  # 1/12 von 1.230 EUR
SOZIAL_PAUSCHALE_RATE = 0.223  # 22,3 % Abzug für Sozialpauschale
BAFOEG_FREIBETRAG = 389.0  # Grundfreibetrag ab Jan 2026

# Sozialversicherung Werkstudent
RV_RATE = 0.093  # Rentenversicherung Anteil Arbeitnehmer (9,3 %)
KV_STUDENTISCH = 135.0  # Ca. Kosten für studentische KV/PV pro Monat

# Wenn du deine aktuelle SHK-Situation exakt abbilden willst:
FU_MONATSSTUNDEN = 41
FU_STUNDENLOHN = 15.08

aktuell_brutto = FU_MONATSSTUNDEN * FU_STUNDENLOHN  # Ergibt exakt 618.28
# Da die RV im Übergangsbereich bei diesem Betrag nur ~2 Euro beträgt:
aktuell_netto_job = aktuell_brutto - 2.03


# ==========================================
# 2. RECHENLOGIK (FUNKTIONEN)
# ==========================================


def estimate_lohnsteuer(brutto_monat):
    """Schätzt die Lohnsteuer für Steuerklasse 1 (2026) unter Berücksichtigung

    des Grundfreibetrags (12.348 EUR) und Vorsorgeaufwendungen.
    """
    brutto_jahr = brutto_monat * 12
    # Grundfreibetrag + Werbungskostenpauschale (1.230 EUR)
    steuerfrei_jahr = 12348 + 1230
    pflicht_rv_jahr = brutto_jahr * RV_RATE  # RV mindert das zu verst. Einkommen

    steuerpflichtig_jahr = max(0, brutto_jahr - steuerfrei_jahr - pflicht_rv_jahr)

    if steuerpflichtig_jahr == 0:
        return 0.0

    # Lineare Annäherung des Eingangssteuersatzes (startet bei 14%) für studentische Einkommen
    effektiver_steuersatz = 0.14 + (0.000004 * steuerpflichtig_jahr)
    effektiver_steuersatz = min(0.42, effektiver_steuersatz)

    steuer_jahr = steuerpflichtig_jahr * effektiver_steuersatz
    return round(steuer_jahr / 12, 2)


def berechne_gesamteinkommen(wochenstunden, stundenlohn, is_werkstudent=True):
    """Berechnet das reale Netto-Einkommen basierend auf der gesetzlichen

    Kombination aus Job-Netto, BAföG-Anrechnung und Buchgeld.
    """
    # 1. Job-Brutto berechnen (Monatsschnitt = Wochenstunden * 4)
    brutto_monat = wochenstunden * stundenlohn * 4.345

    # 2. Abzüge vom Gehalt (Werkstudentenprivileg)
    rv_abzug = brutto_monat * RV_RATE
    steuer_abzug = estimate_lohnsteuer(brutto_monat)
    job_netto_vor_kv = brutto_monat - rv_abzug - steuer_abzug

    # Krankenkassen-Logik
    # Grenze Familienversicherung liegt 2026 bei 565 EUR Gewinn + 102,50 EUR Werbungskosten = 667,50 EUR Brutto
    if is_werkstudent and brutto_monat > 667.50:
        kv_kosten = KV_STUDENTISCH
    else:
        kv_kosten = (
            AKTUELL_KV_EIGEN  # Falls unter der Grenze oder explizit SHK-Szenario
        )

    job_netto_real = job_netto_vor_kv - kv_kosten

    # 3. BAföG-Anrechnung (§ 21ff BAföG)
    einkommen_nach_wk = max(0, brutto_monat - WERBUNGSKOSTEN_PAUSCHALE_MONAT)
    einkommen_nach_sozialpauschale = einkommen_nach_wk * (
        1 - SOZIAL_PAUSCHALE_RATE
    )
    anzurechnendes_einkommen = max(
        0, einkommen_nach_sozialpauschale - BAFOEG_FREIBETRAG
    )

    # Rechter BAföG-Ersatzbetrag nach Kürzung
    bafoeg_netto = max(0, BAFOEG_ERSATZ_MAX - anzurechnendes_einkommen)

    # 4. Gesamtsumme im Monat
    gesamt_netto = job_netto_real + bafoeg_netto + BUCHGELD

    return {
        "Brutto_Job": round(brutto_monat, 2),
        "Netto_Job": round(job_netto_real, 2),
        "Stipendium_Kuerzung": round(
            BAFOEG_ERSATZ_MAX - bafoeg_netto, 2
        ),  # Wie viel wird abgezogen?
        "Stipendium_Rest": round(bafoeg_netto + BUCHGELD, 2),
        "Gesamt_Netto": round(gesamt_netto, 2),
    }

# Aktuellen Status berechnen (10h SHK)
aktuell = berechne_gesamteinkommen(AKTUELL_STUNDEN, AKTUELL_LOHN, is_werkstudent=False
)
print("-" * 60)
print(f"DEINE AKTUELLE SHK-SITUATION (ca. 10h/Woche):")
print(f"Job-Brutto: {aktuell['Brutto_Job']} €")
print(f"Stipendium (Ersatz + Buchgeld): {aktuell['Stipendium_Rest']} €")
print(f"--> EFFEKTIVES NETTO AKTUELL: {aktuell['Gesamt_Netto']} €")
print("-" * 60 + "\n")

# Simulations-Bereiche definieren
stunden_range = [12, 14, 16, 18, 20]  # Mögliche Werkstudenten-Stunden
lohn_range = [16, 18, 20, 21.5, 23, 25]  # Mögliche Stundenlöhne

# Matrix für den Gesamt-Netto-Vergleich aufbauen
matrix_data = {}
for h in stunden_range:
    column_cells = []
    for l in lohn_range:
        res = berechne_gesamteinkommen(h, l, is_werkstudent=True)
        column_cells.append(res["Gesamt_Netto"])
    matrix_data[f"{h} Std/Woche"] = column_cells

df_vergleich = pd.DataFrame(matrix_data, index=[f"{l} €/Std" for l in lohn_range])

print("VERGLEICHS-MATRIX: DEIN TOTALES MONATS-NETTO ALS WERKSTUDENT")
print("(Inklusive Job-Netto, gekürztem Stipendium & Buchgeld, abzüglich KV)")
display(df_vergleich)

print("\n" + "-" * 60)
print(f"Jeder Wert, der UNTER {aktuell['Gesamt_Netto']} € liegt, ist ein Verlustgeschäft")
print("verglichen mit deiner aktuellen SHK-Stelle bei doppeimport numpy as np")
import pandas as pd

# ==========================================
# 1. GESETZLICHE PARAMETER (STAND 2026) & INPUTS
# ==========================================

# Stipendium (SDW) / BAföG-Sätze
BAFOEG_ERSATZ_MAX = 855.0  # Maximaler BAföG-Ersatzbetrag
BUCHGELD = 300.0  # Einkommensunabhängiges Buchgeld

# BAföG-Anrechnungslogik (§ 21, 23 BAföG)
WERBUNGSKOSTEN_PAUSCHALE_MONAT = 102.50  # 1/12 von 1.230 EUR
SOZIAL_PAUSCHALE_RATE = 0.223  # 22,3 % Abzug für Sozialpauschale
BAFOEG_FREIBETRAG = 389.0  # Grundfreibetrag ab Jan 2026

# Sozialversicherung Werkstudent
RV_RATE = 0.093  # Rentenversicherung Anteil Arbeitnehmer (9,3 %)
KV_STUDENTISCH = 135.0  # Ca. Kosten für studentische KV/PV pro Monat

# === Deine aktuellen SHK-Eckdaten ===
FU_MONATSSTUNDEN = 41
FU_STUNDENLOHN = 15.08
AKTUELL_STUNDEN_WOCHE = 9.44  # 41 Monatsstunden / 4.345 Wochen im Monat
AKTUELL_KV_EIGEN = 0.0  # Da unter der Grenze (Familienversichert oder über Job)

aktuell_brutto = FU_MONATSSTUNDEN * FU_STUNDENLOHN  # Ergibt exakt 618.28
aktuell_netto_job = aktuell_brutto - 2.03


# ==========================================
# 2. RECHENLOGIK (FUNKTIONEN)
# ==========================================


def estimate_lohnsteuer(brutto_monat):
    """Schätzt die Lohnsteuer für Steuerklasse 1 (2026) unter Berücksichtigung

    des Grundfreibetrags (12.348 EUR) und Vorsorgeaufwendungen.
    """
    brutto_jahr = brutto_monat * 12
    steuerfrei_jahr = 12348 + 1230
    pflicht_rv_jahr = brutto_jahr * RV_RATE

    steuerpflichtig_jahr = max(0, brutto_jahr - steuerfrei_jahr - pflicht_rv_jahr)

    if steuerpflichtig_jahr == 0:
        return 0.0

    effektiver_steuersatz = 0.14 + (0.000004 * steuerpflichtig_jahr)
    effektiver_steuersatz = min(0.42, effektiver_steuersatz)

    steuer_jahr = steuerpflichtig_jahr * effektiver_steuersatz
    return round(steuer_jahr / 12, 2)


def berechne_gesamteinkommen(wochenstunden, stundenlohn, is_werkstudent=True):
    """Berechnet das reale Netto-Einkommen basierend auf der gesetzlichen

    Kombination aus Job-Netto, BAföG-Anrechnung und Buchgeld.
    """
    # 1. Job-Brutto berechnen (Monatsschnitt = Wochenstunden * 4.345)
    brutto_monat = wochenstunden * stundenlohn * 4.345

    # 2. Abzüge vom Gehalt
    rv_abzug = brutto_monat * RV_RATE
    steuer_abzug = estimate_lohnsteuer(brutto_monat)
    job_netto_vor_kv = brutto_monat - rv_abzug - steuer_abzug

    # Krankenkassen-Logik
    if is_werkstudent and brutto_monat > 667.50:
        kv_kosten = KV_STUDENTISCH
    else:
        kv_kosten = AKTUELL_KV_EIGEN

    job_netto_real = job_netto_vor_kv - kv_kosten

    # 3. BAföG-Anrechnung (§ 21ff BAföG)
    einkommen_nach_wk = max(0, brutto_monat - WERBUNGSKOSTEN_PAUSCHALE_MONAT)
    einkommen_nach_sozialpauschale = einkommen_nach_wk * (
        1 - SOZIAL_PAUSCHALE_RATE
    )
    anzurechnendes_einkommen = max(
        0, einkommen_nach_sozialpauschale - BAFOEG_FREIBETRAG
    )

    bafoeg_netto = max(0, BAFOEG_ERSATZ_MAX - anzurechnendes_einkommen)

    # 4. Gesamtsumme im Monat
    gesamt_netto = job_netto_real + bafoeg_netto + BUCHGELD

    return {
        "Brutto_Job": round(brutto_monat, 2),
        "Netto_Job": round(job_netto_real, 2),
        "Stipendium_Kuerzung": round(BAFOEG_ERSATZ_MAX - bafoeg_netto, 2),
        "Stipendium_Rest": round(bafoeg_netto + BUCHGELD, 2),
        "Gesamt_Netto": round(gesamt_netto, 2),
    }


# ==========================================
# 3. AUSWERTUNG & SIMULATION
# ==========================================

# Aktuellen Status berechnen (mit den korrekten FU-Variablen)
aktuell = berechne_gesamteinkommen(
    wochenstunden=AKTUELL_STUNDEN_WOCHE,
    stundenlohn=FU_STUNDENLOHN,
    is_werkstudent=False,
)

print("-" * 60)
print(f"DEINE AKTUELLE SHK-SITUATION (ca. 10h/Woche):")
print(f"Job-Brutto: {aktuell['Brutto_Job']} €")
print(f"Stipendium (Ersatz + Buchgeld): {aktuell['Stipendium_Rest']} €")
print(f"--> EFFEKTIVES NETTO AKTUELL: {aktuell['Gesamt_Netto']} €")
print("-" * 60 + "\n")

# Simulations-Bereiche definieren
stunden_range = [12, 14, 16, 18, 20]  # Mögliche Werkstudenten-Stunden
lohn_range = [16, 18, 20, 21.5, 23, 25]  # Mögliche Stundenlöhne

# Matrix für den Gesamt-Netto-Vergleich aufbauen
matrix_data = {}
for h in stunden_range:
    column_cells = []
    for l in lohn_range:
        res = berechne_gesamteinkommen(h, l, is_werkstudent=True)
        column_cells.append(res["Gesamt_Netto"])
    matrix_data[f"{h} Std/Woche"] = column_cells  # <-- Wichtig: Muss unter dem 'for l...' eingerückt sein!

df_vergleich = pd.DataFrame(matrix_data, index=[f"{l} €/Std" for l in lohn_range])

print("VERGLEICHS-MATRIX: DEIN TOTALES MONATS-NETTO ALS WERKSTUDENT")
print("(Inklusive Job-Netto, gekürztem Stipendium & Buchgeld, abzüglich KV)")
display(df_vergleich)

print("\n" + "-" * 60)
print(
    f"Jeder Wert, der UNTER {aktuell['Gesamt_Netto']} € liegt, ist ein Verlustgeschäft"
)
print("verglichen mit deiner aktuellen SHK-Stelle bei doppelter Arbeit!")
print("-" * 60)

NameError: name 'AKTUELL_STUNDEN' is not defined

In [ ]:
616.25/41

15.03048780487805